# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

# Ejecuta desde la carpeta del proyecto o desde la carpeta que contiene el notebook.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "processed" / "train.csv").exists():
    # Compatibilidad si el notebook se abre desde una carpeta hermana del proyecto Challenge 1.
    candidates = [
        PROJECT_DIR / "Challenge 1",
        PROJECT_DIR / "challenge_1",
        PROJECT_DIR / "wine_data_challenge",
        PROJECT_DIR.parent / "wine_data_challenge",
    ]
    for candidate in candidates:
        if (candidate / "data" / "processed" / "train.csv").exists():
            PROJECT_DIR = candidate
            break

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Los árboles no requieren escalado: imputamos con la mediana.
preprocessor_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Logistic Regression y KNN sí se benefician del escalado.
preprocessor_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


**Respuesta:** `StandardScaler` debe estar dentro del pipeline porque, en cada partición de validación cruzada, el escalador debe aprender media y desviación estándar solo con el fold de entrenamiento. Si se ajusta antes de la validación cruzada, se filtra información de los folds de validación (*data leakage*), lo que puede dar una estimación demasiado optimista del rendimiento. El pipeline garantiza que el preprocesamiento se ajuste de nuevo dentro de cada fold.\n

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", KNeighborsClassifier())
    ])
}

print("Modelos candidatos:", list(models.keys()))


Modelos candidatos: ['DecisionTree', 'LogisticRegression', 'KNN']


## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

initial_rows = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=cv, scoring="accuracy",
        return_train_score=True, n_jobs=-1
    )
    initial_rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "validation_accuracy_mean": scores["test_score"].mean(),
        "validation_accuracy_std": scores["test_score"].std()
    })

initial_comparison = pd.DataFrame(initial_rows).sort_values(
    "validation_accuracy_mean", ascending=False
).reset_index(drop=True)

display(initial_comparison)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
initial_comparison.to_csv(REPORTS_DIR / "initial_comparison.csv", index=False)


,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std
0,LogisticRegression,1.0000,0.979064,0.027713
1,KNN,0.9771,0.950739,0.035274
2,DecisionTree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "LogisticRegression": {
        "model__C": [0.1, 1, 10],
        "model__solver": ["lbfgs"]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    }
}

searches = {}
tuned_rows = []
for name, model in models.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_rows.append({
        "model": name,
        "best_cv_accuracy": search.best_score_,
        "train_accuracy_at_best": search.cv_results_["mean_train_score"][search.best_index_],
        "best_params": json.dumps(search.best_params_)
    })

tuned_comparison = pd.DataFrame(tuned_rows).sort_values(
    "best_cv_accuracy", ascending=False
).reset_index(drop=True)

display(tuned_comparison)
tuned_comparison.to_csv(REPORTS_DIR / "tuned_comparison.csv", index=False)


,model,best_cv_accuracy,train_accuracy_at_best,best_params
0,LogisticRegression,0.979064,1.000000,"{""model__C"": 1, ""model__solver"": ""lbfgs""}"
1,KNN,0.957882,0.964773,"{""model__n_neighbors"": 7, ""model__weights"": ""u..."
2,DecisionTree,0.902463,0.985934,"{""model__max_depth"": null, ""model__min_samples..."


## 6. Selección y persistencia del campeón

In [8]:
# Elegir el modelo con mayor accuracy promedio de validación cruzada.
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name]
champion_model = champion.best_estimator_

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(champion_model, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion_name": champion_name,
    "selection_metric": "accuracy",
    "best_cv_accuracy": float(champion.best_score_),
    "best_params": champion.best_params_,
    "cv_splits": 5,
    "random_state": RANDOM_STATE,
    "target": contract["target"],
    "features": feature_columns
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Consolidar los resultados de todos los modelos y configuraciones probadas.
all_results = []
for name, search in searches.items():
    result = pd.DataFrame(search.cv_results_)
    result["model"] = name
    all_results.append(result)
cv_results = pd.concat(all_results, ignore_index=True)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print("Modelo campeón:", champion_name)
print(f"Accuracy promedio CV: {champion.best_score_:.4f}")
print("Mejores parámetros:", champion.best_params_)


Modelo campeón: LogisticRegression
Accuracy promedio CV: 0.9791
Mejores parámetros: {'model__C': 1, 'model__solver': 'lbfgs'}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

La selección se basa en el mayor `best_score_` de validación cruzada estratificada de 5 folds, sin cargar ni utilizar `test.csv`. En esta ejecución, Logistic Regression obtuvo el mejor promedio de accuracy (aprox. 0,979). Su desviación estándar en la comparación inicial fue baja (aprox. 0,028), por lo que su desempeño fue relativamente estable entre folds.

La accuracy de entrenamiento del mejor Logistic Regression llega a 1,00, por encima de la validación, lo que sugiere una posible diferencia de generalización que debe vigilarse. Aun así, su validación cruzada supera a KNN y Decision Tree en esta búsqueda. Logistic Regression también suele ser más interpretable que KNN, aunque sus coeficientes requieren considerar el escalado de las variables. El conjunto de prueba se mantiene reservado para una evaluación final imparcial.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.
